# 00 · LLM setup (NVIDIA NIM / Nemotron)

Checks that the environment and API key work. Config lives in `codeverse/.env` (gitignored).

| var | meaning |
|---|---|
| `NVIDIA_API_KEY` | your `nvapi-…` key |
| `LLM_BASE_URL` | OpenAI-compatible endpoint |
| `LLM_MODEL` | default model for `chat()` |

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from codeverse import llm

key = os.getenv('NVIDIA_API_KEY', '')
print('key   :', key[:9] + '…' + key[-4:] if key else 'MISSING')
print('url   :', os.getenv('LLM_BASE_URL'))
print('model :', llm.default_model())

## Available models

In [ ]:
llm.list_models('nemotron')

## Plain chat

In [ ]:
print(llm.chat('In one sentence, what is a software hotspot (churn x complexity)?'))

## Streaming

In [ ]:
for tok in llm.stream('Name 5 planets-as-code metaphors for a 3D repo visualizer, one line each.'):
    print(tok, end='', flush=True)

## Code generation + test loop

Ask the model for code, extract the fenced block, run it in an isolated namespace, then run a quick test.

In [ ]:
SYSTEM = (
    'You are a senior Python engineer. Return ONLY one ```python fenced block. '
    'No explanations. Standard library only unless told otherwise.'
)

def generate(task: str, **kw) -> str:
    return llm.extract_code(llm.chat(task, system=SYSTEM, **kw), 'python')

src = generate(
    'Write a function stable_hash_angle(path: str) -> float that maps a file path to a '
    'deterministic angle in [0, 2*pi) using hashlib.blake2b. Same input must always give the same output.'
)
print(src)

In [ ]:
ns: dict = {}
exec(src, ns)
f = ns['stable_hash_angle']

import math
a = f('fastapi/routing.py')
assert a == f('fastapi/routing.py'), 'not deterministic'
assert 0 <= a < 2 * math.pi, 'out of range'
assert f('a.py') != f('b.py'), 'suspicious collisions'
print('✅ generated code passed:', a)

> ⚠️ `exec` runs model output directly. Fine for small scratch experiments; don't point it at anything that touches the filesystem or network.